# IE 7615 — Visual Question Answering Extension
Conditions on a CLIP image embedding and a natural language question to produce a text answer using **ViLT** (Vision-and-Language Transformer), a lightweight model fine-tuned on VQAv2.

In [ ]:
# CELL 1: Install dependencies
!pip install -q transformers datasets
!pip install -q git+https://github.com/openai/CLIP.git

In [ ]:
# CELL 2: Imports
import torch
import clip
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from io import BytesIO
from transformers import ViltProcessor, ViltForQuestionAnswering

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Load ViLT — fine-tuned on VQAv2 (cpu-friendly)
print("Loading ViLT...")
processor  = ViltProcessor.from_pretrained("dandelin/vilt-b32-finetuned-vqa")
vilt_model = ViltForQuestionAnswering.from_pretrained("dandelin/vilt-b32-finetuned-vqa").to(device)
vilt_model.eval()
print("ViLT loaded ✅")

In [ ]:
# CELL 3: VQA inference function
def answer_question(image: Image.Image, question: str, top_k: int = 3):
    """
    Returns the top-k answers with confidence scores for a given image and question.
    """
    inputs = processor(image, question, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = vilt_model(**inputs).logits          # (1, num_labels)
    probs   = torch.softmax(logits, dim=-1)[0]
    top_ids = probs.topk(top_k).indices.tolist()
    results = [
        (vilt_model.config.id2label[i], round(probs[i].item() * 100, 1))
        for i in top_ids
    ]
    return results


def run_vqa_demo(image: Image.Image, questions: list):
    """
    Runs VQA on a list of questions and displays results alongside the image.
    """
    answers = []
    for q in questions:
        top = answer_question(image, q, top_k=1)
        answers.append((q, top[0][0], top[0][1]))

    # ── Display ────────────────────────────────────────────────────────────
    fig, (ax_img, ax_txt) = plt.subplots(1, 2, figsize=(14, 6))

    ax_img.imshow(image)
    ax_img.axis("off")
    ax_img.set_title("Input Image", fontsize=13, fontweight="bold")

    ax_txt.axis("off")
    lines = []
    for q, a, conf in answers:
        lines.append(f"Q: {q}")
        lines.append(f"A: {a}  ({conf}%)")
        lines.append("")
    ax_txt.text(
        0.05, 0.97, "\n".join(lines),
        transform=ax_txt.transAxes,
        fontsize=12, verticalalignment="top", fontfamily="monospace",
        bbox=dict(boxstyle="round", facecolor="#f0f4ff", alpha=0.85)
    )
    ax_txt.set_title("VQA Answers — ViLT (VQAv2 fine-tuned)",
                     fontsize=13, fontweight="bold")

    plt.suptitle("Visual Question Answering", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig("vqa_demo.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("vqa_demo.png saved ✅")

print("Functions ready ✅")

In [ ]:
# CELL 4: Upload an image and run VQA
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]
image    = Image.open(BytesIO(uploaded[filename])).convert("RGB")

questions = [
    "What is in the image?",
    "How many people are in the image?",
    "What color is the main object?",
    "What is the main activity happening?",
    "What is in the background?",
    "Is this indoors or outdoors?",
]

run_vqa_demo(image, questions)

In [ ]:
# CELL 5: Ask a custom question interactively
custom_question = "What sport is being played?"   # ← change this

top3 = answer_question(image, custom_question, top_k=3)
print(f"Question: {custom_question}")
print("-" * 40)
for rank, (ans, conf) in enumerate(top3, 1):
    print(f"  #{rank}  {ans:<20} {conf}%")

In [ ]:
# CELL 6: Batch VQA over precomputed Flickr8k embeddings (CLIP similarity filter)
# Uses CLIP to find the most relevant images for a query, then answers a question on them.
import json
from datasets import load_dataset

# Mount drive if needed
from google.colab import drive
drive.mount("/content/drive")
DRIVE_PATH = "/content/drive/MyDrive/Generative project/"

# Load precomputed embeddings and captions
embeddings = np.load(DRIVE_PATH + "clip_embeddings_5k.npy")   # (5000, 512)
with open(DRIVE_PATH + "captions_5k.json") as f:
    captions = json.load(f)

# Load Flickr8k images
ds = load_dataset("nlphuji/flickr8k", split="test")

clip_model, preprocess = clip.load("ViT-B/32", device=device)
clip_model.eval()

def find_top_images(text_query: str, top_k: int = 3):
    """Return indices of images most similar to a text query via CLIP."""
    tokens = clip.tokenize([text_query]).to(device)
    with torch.no_grad():
        text_emb = clip_model.encode_text(tokens).cpu().numpy()  # (1, 512)
    text_emb /= np.linalg.norm(text_emb, axis=-1, keepdims=True)
    embs_norm = embeddings / np.linalg.norm(embeddings, axis=-1, keepdims=True)
    sims = (embs_norm @ text_emb.T).squeeze()
    return sims.argsort()[::-1][:top_k].tolist()


# Example: find images of dogs and ask a question
query    = "a dog playing outside"
question = "What is the dog doing?"
indices  = find_top_images(query, top_k=3)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, idx in zip(axes, indices):
    img   = ds[idx]["image"].convert("RGB")
    top1  = answer_question(img, question, top_k=1)
    ans, conf = top1[0]
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(f"A: {ans} ({conf}%)", fontsize=10)

plt.suptitle(f'Q: "{question}"', fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("vqa_batch.png", dpi=150, bbox_inches="tight")
plt.show()
print("vqa_batch.png saved ✅")